# 🔗 STACKING MULTIMODAL — **v2 AVANZADO** (blend óptimo · MoE · selección por etiqueta)
## TFM · Fusión de las tres modalidades · Universidad de Salamanca

---

Versión avanzada de la fusión: **blending con pesos optimizados** (en probabilidad, en logit y sobre
las probabilidades sin calibrar), **stacking enriquecido**, **Mixture-of-Experts** con *gating* por
paciente, **ensemble de fusores**, **selección del mejor fusor por etiqueta** y **Caruana**.

## ⚠️ Requisito de ejecución
Necesita los CSV de las **tres** modalidades (`salidas/01_cxr/v2`, `02_ecg/v2`, `03_labs/v2`).
Va **el último** de todo el pipeline.

## 🔄 Adaptación a las conclusiones del EDA

| Cambio | Detalle | Origen |
|---|---|---|
| **Etiquetado FINAL** | `POS=(==1)` · `NEG=(==0)|(NaN→0)` · **−1 ENMASCARADO**. | §3 |
| **Métrica primaria = AUC-PR** | `macroP` ahora devuelve **AP**, no ROC, y es quien **elige el fusor ganador**. `macroP_auc` conserva el ROC como secundario. | §3 |
| **IC bootstrap** | `ci_macroP` para el macro AP; se declara si los IC **se solapan**. | §1 |
| **Fusión vs mejor mono** | Bloque explícito con veredicto y JSON. | §11 · §5 |
| **Prevalencia** | `aps` devuelve (AP, prevalencia) por etiqueta: la prevalencia **es la línea base** de la AP. | §3 |

### ⚠️ Dos correcciones de fondo en este notebook
1. **Etiquetado**: `tgt()` usaba `policy="zeros"` (el −1 contaba como **negativo** y no se
   enmascaraba) y **derivaba siempre** negativos de *No Finding*, sin opción de desactivarlo. Era
   incoherente con las tres modalidades. Corregido al etiquetado FINAL.
2. **Métrica**: el notebook **solo importaba `roc_auc_score`**. La elección del fusor ganador se hacía
   por AUC-ROC. Ahora se hace por **AUC-PR**, que es la métrica primaria del proyecto.

> Consecuencia de ambas: **las cifras de fusión anteriores no son comparables**. Hay que reejecutar.
> El fusor ganador puede cambiar respecto a la selección por ROC; si cambia, explicarlo en la memoria.

- **`cxr_view` NUNCA como predictor**: no entra en el contexto del fusor.

---

## 📌 Para la documentación posterior (recuadros naranjas a redactar con los resultados)

- **LA PREGUNTA CENTRAL**: ¿supera la fusión a la mejor modalidad en solitario? El notebook lo
  responde con IC y guarda el veredicto en `veredicto_fusion_vs_mono.json`. **Si los IC se solapan,
  la mejora no es concluyente** — escribirlo tal cual, no maquillarlo.
- **Contexto ya medido**: LABS v1 = 0,3968 [0,368–0,436] · LABS v2 = 0,3910 [0,362–0,429] ·
  ECG v1 = 0,3667 [0,339–0,405]. En tabular, **v2 no superó a v1**: el techo de una modalidad se
  alcanza pronto, y eso hace más exigente la comparación para la fusión.
- **¿Compensa la complejidad?** Este notebook prueba 10 fusores. Si el mejor no bate al **blending
  simple** por más que el ruido, la conclusión honesta es que **la sofisticación no aporta** — y es
  un resultado tan publicable como el contrario.
- **Aportación diferencial por patología** (§5): la imagen debería liderar Atelectasia y Opacidad;
  el tabular aportar en Edema, Derrame y Cardiomegalia. La selección **por etiqueta** es justo el
  fusor que puede explotar eso: mirar qué modalidad elige en cada una y si coincide con la fisiología.
- **Calibración**: tras fusionar hay que **recalibrar** (la calibración por modalidad no se conserva).
- **Equidad**: subgrupos con n<60 ⇒ ruido, no inequidad.



In [ ]:
# CELDA 1 · DEPENDENCIAS
import subprocess, sys
for pkg in ["scikit-learn","scipy","pandas","numpy","matplotlib","torch"]:
    subprocess.run([sys.executable,"-m","pip","install",pkg,"-q"],check=False)
print("Dependencias listas.")

In [ ]:
# CELDA 2 · IMPORTS / CONSTANTES / OBJETIVOS
import json, warnings, copy
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
import torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings("ignore"); SEED=42; np.random.seed(SEED); torch.manual_seed(SEED)
NB=Path.cwd(); BASE=Path(r"C:\TFM\1.Opción - Symile Mimic\symile-mimic-a-multimodal-clinical-dataset-of-chest-x-rays-electrocardiograms-and-blood-labs-from-mimic-iv-1.0.0")
CSV=BASE/"data_csv"/"clean"
CXR=Path(r"C:\TFM\1.Opción - Symile Mimic\tfm_multimodal_clinico\salidas\01_cxr\v2"); ECG=Path(r"C:\TFM\1.Opción - Symile Mimic\tfm_multimodal_clinico\salidas\02_ecg\v2"); LABS=Path(r"C:\TFM\1.Opción - Symile Mimic\tfm_multimodal_clinico\salidas\03_labs\v2")
OUT=Path(r"C:\TFM\1.Opción - Symile Mimic\tfm_multimodal_clinico\salidas\04_stacking\v2"); OUT.mkdir(exist_ok=True); FG=OUT/"figuras"; FG.mkdir(exist_ok=True)
LABELS=["Atelectasis","Cardiomegaly","Edema","Lung Opacity","No Finding","Pleural Effusion"]; N=len(LABELS)
NF="No Finding"; PATH=[l for l in LABELS if l!=NF]; CORE=["Cardiomegaly","Edema","Pleural Effusion"]; MODS=["CXR","ECG","LABS"]
df_tr=pd.read_csv(CSV/"train_clean.csv",sep=";"); df_vl=pd.read_csv(CSV/"val_clean.csv",sep=";"); df_te=pd.read_csv(CSV/"test_clean.csv",sep=";")
# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · tgt  (objetivos con el ETIQUETADO FINAL)
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : convierte los estados 1/0/−1/NaN en (y, m): y=0-1, m=1 si la etiqueta cuenta.
# POR QUÉ    : el fusor debe usar EXACTAMENTE la misma definición que las tres modalidades; si no,
#              combinaría probabilidades entrenadas contra un objetivo distinto del que se evalúa.
#              POS=(==1) · NEG=(==0)|(NaN→0) · −1 ENMASCARADO (U-Ignore).
# ENTRADAS   : df (split) · policy ("ignore")
# SALIDAS    : (y (n,6) float32, m (n,6) float32)
# ORIGEN EDA : §3 · "negativo = 0 + NaN→0; el −1 se enmascara; «Sin hallazgo» NO como negativo".
# ⚠️ CAMBIO vs la versión previa de ESTE notebook: usaba policy="zeros" (el −1 como negativo, sin
#    enmascarar) y derivaba SIEMPRE negativos de «No Finding», sin opción de desactivarlo.
# INTERPRETACIÓN FUTURA (→ recuadro naranja): las cifras de fusión anteriores NO son comparables.
# ══════════════════════════════════════════════════════════════════════════════
def tgt(df, policy="ignore"):
    raw = df[LABELS].to_numpy(float); n = raw.shape[0]
    y = (raw == 1.0).astype(np.float32)          # 1 → positivo ; 0 y NaN → 0 (negativo)
    m = np.ones((n, N), np.float32)              # por defecto TODO entra en pérdida/métrica
    if policy == "ignore": m[raw == -1.0] = 0.0  # −1 → ENMASCARADO (definición FINAL)
    return y, m

y_tr,m_tr=tgt(df_tr); y_vl,m_vl=tgt(df_vl); y_te,m_te=tgt(df_te)
print("targets:",y_tr.shape,y_vl.shape,y_te.shape)


In [ ]:
# CELDA 3 · CARGAR LAS 3 MODALIDADES (calibradas Y raw) + contexto
def load(d,pref,split,df,cal=True):
    c=pd.read_csv(d/f"{pref}_pred_{split}.csv").set_index("hadm_id"); suf="_cal" if cal else ""
    cols=[f"{pref}_{l.replace(' ','_')}{suf}" for l in LABELS]
    return c.reindex(df["hadm_id"].to_numpy())[cols].fillna(0.5).to_numpy(np.float32)
def base(split,df,cal=True): return {"CXR":load(CXR,"cxr",split,df,cal),"ECG":load(ECG,"ecg",split,df,cal),"LABS":load(LABS,"labs",split,df,cal)}
Bc={"tr":base("oof_train",df_tr),"vl":base("val",df_vl),"te":base("test",df_te)}
Br={"tr":base("oof_train",df_tr,False),"vl":base("val",df_vl,False),"te":base("test",df_te,False)}
def ctx(df):
    G={0:0,1:1,"0":0,"1":1,"M":1,"F":0}; R={"UNKNOWN":0,"WHITE":1,"BLACK":2,"ASIAN":3,"HISPANIC_LATINO":4,"OTHER_KNOWN":0}
    A={"SCHEDULED":0,"EMERGENCY":1,"OBSERVATION":2,"URGENT":3}; L={"EMERGENCY_ROOM":0,"REFERRAL":1,"TRANSFER":2,"INTRA_HOSPITAL":3}
    n=len(df); cols=[df["age"].astype(float).to_numpy()[:,None]/100, df["gender"].map(lambda v:float(G.get(v,0))).to_numpy()[:,None]]
    def oh(s,mp):
        M=np.zeros((n,max(mp.values())+1),np.float32)
        for i,v in enumerate(s): M[i,mp.get(str(v).upper(),0)]=1
        return M
    for c,mp in [("race",R),("admission_type",A),("admission_location",L)]: cols.append(oh(df[c],mp))
    return np.hstack(cols).astype(np.float32)
Ctr,Cvl,Cte=ctx(df_tr),ctx(df_vl),ctx(df_te)
# ════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · macroP  (MÉTRICA PRIMARIA = AUC-PR macro sobre las 5 patologías)
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : promedia el Average Precision de las 5 patologías (excluye «Sin hallazgo»).
# POR QUÉ    : con desbalanceo, la AP es más honesta que el ROC. Esta función gobierna la ELECCIÓN
#              del mejor fusor, así que es la decisión más importante del notebook.
# ORIGEN EDA : §3 · "métrica primaria AUC-PR por patología; secundaria AUC-ROC".
# ⚠️ CAMBIO: antes devolvía AUC-ROC. Ahora devuelve AUC-PR; `macroP_auc` conserva el ROC secundario.
# INTERPRETACIÓN FUTURA (→ recuadro naranja): el fusor elegido puede cambiar respecto a la selección
#              por ROC. Si cambia, explicarlo: la AP prioriza la precisión en la clase minoritaria.
# ══════════════════════════════════════════════════════════════════════════════
def macroP(p, y, m):
    a = []
    for j, l in enumerate(LABELS):
        if l == NF: continue
        s = m[:, j] == 1; yt = y[s, j]
        if yt.sum() < 2 or (1 - yt).sum() < 2: continue
        a.append(average_precision_score(yt, p[s, j]))
    return float(np.mean(a))

def macroP_auc(p, y, m):
    """AUC-ROC macro sobre patologías — métrica SECUNDARIA (se mantiene para comparar con el histórico)."""
    a = []
    for j, l in enumerate(LABELS):
        if l == NF: continue
        s = m[:, j] == 1; yt = y[s, j]
        if yt.sum() < 2 or (1 - yt).sum() < 2: continue
        a.append(roc_auc_score(yt, p[s, j]))
    return float(np.mean(a))

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · aps  (AP y prevalencia por etiqueta)
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : AP de cada etiqueta junto a su PREVALENCIA (que es la línea base de la AP).
# ORIGEN EDA : §3 · "la prevalencia condiciona la línea base de la AP: no comparar AP entre
#              patologías de distinta prevalencia".
# ══════════════════════════════════════════════════════════════════════════════
def aps(p, y, m):
    out = {}
    for j, l in enumerate(LABELS):
        s = m[:, j] == 1; yt = y[s, j]
        if yt.sum() < 2 or (1 - yt).sum() < 2: out[l] = (np.nan, np.nan); continue
        out[l] = (average_precision_score(yt, p[s, j]), float(yt.mean()))
    return out

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · ci_macroP  (IC bootstrap del AUC-PR macro)
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : remuestrea PACIENTES con reemplazo y recalcula el macro AP, devolviendo el IC percentil.
# POR QUÉ    : es lo que permite decir si un fusor supera de verdad a otro, o si la diferencia es ruido.
# ORIGEN EDA : §1 · "val (750) y test (464) son pequeños → IC bootstrap SIEMPRE".
# INTERPRETACIÓN FUTURA (→ recuadro naranja): si el IC del mejor fusor SE SOLAPA con el del mejor
#              mono-modelo, la mejora NO es concluyente y hay que decirlo explícitamente.
# ══════════════════════════════════════════════════════════════════════════════
def ci_macroP(p, y, m, n_boot=300, alpha=0.05, seed=SEED):
    rng = np.random.RandomState(seed); vals = []
    for _ in range(n_boot):
        bs = rng.randint(0, len(y), len(y)); a = []
        for j, l in enumerate(LABELS):
            if l == NF: continue
            s = m[bs][:, j] == 1; yt = y[bs][s, j]
            if yt.sum() < 2 or (1 - yt).sum() < 2: continue
            a.append(average_precision_score(yt, p[bs][s, j]))
        if a: vals.append(np.mean(a))
    return (float(np.percentile(vals, 100*alpha/2)), float(np.percentile(vals, 100*(1-alpha/2)))) if vals else (np.nan, np.nan)

def aucs(p,y,m):
    # AP por etiqueta (ESCALAR). El resto del notebook espera un dict {etiqueta: valor}.
    return {l:(average_precision_score(y[m[:,j]==1,j],p[m[:,j]==1,j]) if (y[m[:,j]==1,j].sum()>=2 and (1-y[m[:,j]==1,j]).sum()>=2) else float("nan")) for j,l in enumerate(LABELS)}
logit=lambda p: np.log(np.clip(p,1e-4,1-1e-4)/(1-np.clip(p,1e-4,1-1e-4)))
def stack3(B,key): return np.stack([B[key]["CXR"],B[key]["ECG"],B[key]["LABS"]],0)
print("Modalidades cargadas (cal + raw) y contexto:",Ctr.shape[1],"dims")





In [ ]:
# CELDA 4 · BLENDING ÓPTIMO: probabilidad, logit y raw (pesos por etiqueta sobre OOF)
def fit_blend(Btr_key, space="prob"):
    S=stack3(Bc,"tr") if Btr_key=="cal" else stack3(Br,"tr")
    if space=="logit": S=logit(S)
    W=np.zeros((N,3))
    for j in range(N):
        s=m_tr[:,j]==1; yt=y_tr[s,j]
        if len(np.unique(yt))<2: W[j]=[1,0,0]; continue
        P3=S[:,s,j]
        def neg(z):
            w=np.exp(z-z.max()); w=w/w.sum(); return -roc_auc_score(yt,w@P3)
        best=None
        for init in [np.array([2.,0,0]),np.array([0.,0,0]),np.array([1.,1,0])]:
            r=minimize(neg,init,method="Nelder-Mead",options={"xatol":1e-3,"fatol":1e-4,"maxiter":400})
            if best is None or r.fun<best.fun: best=r
        z=best.x; w=np.exp(z-z.max()); W[j]=w/w.sum()
    return W
def apply_blend(W, src, key, space="prob"):
    S=stack3(src,key)
    if space=="logit": S=logit(S)
    return np.stack([W[j]@S[:,:,j] for j in range(N)],1)
W_prob=fit_blend("cal","prob"); W_logit=fit_blend("cal","logit"); W_raw=fit_blend("raw","prob")
P={}
P["blend_opt(v3)"]=(apply_blend(W_prob,Bc,"vl"),apply_blend(W_prob,Bc,"te"))
P["blend_logit"]=(apply_blend(W_logit,Bc,"vl","logit"),apply_blend(W_logit,Bc,"te","logit"))
P["blend_raw"]=(apply_blend(W_raw,Br,"vl"),apply_blend(W_raw,Br,"te"))
print("blends listos · logit/raw vs prob:",{k:round(macroP(P[k][1],y_te,m_te),4) for k in P})

In [ ]:
# CELDA 5 · STACKING LogReg (básico) y ENRIQUECIDO (cal+raw+confianza+entropía)
def conf_ent(B,key):
    feats=[]
    for mod in MODS:
        p=B[key][mod]; feats.append(np.abs(p-0.5)); feats.append(-(p*np.log(p+1e-6)+(1-p)*np.log(1-p+1e-6)))
    return np.hstack(feats)
def meta(Xtr,Xvl,Xte,C=0.05):
    sc=StandardScaler().fit(Xtr); a,b,c=sc.transform(Xtr),sc.transform(Xvl),sc.transform(Xte)
    Pv=np.full((len(Xvl),N),0.5,np.float32); Pt=np.full((len(Xte),N),0.5,np.float32)
    for j in range(N):
        s=m_tr[:,j]==1; yj=y_tr[s,j]
        if len(np.unique(yj))<2: continue
        clf=LogisticRegression(C=C,class_weight="balanced",max_iter=3000).fit(a[s],yj)
        Pv[:,j]=clf.predict_proba(b)[:,1]; Pt[:,j]=clf.predict_proba(c)[:,1]
    return Pv,Pt
X18=lambda B,key: np.hstack([B[key]["CXR"],B[key]["ECG"],B[key]["LABS"]])
P["stack_LR"]=meta(np.hstack([X18(Bc,"tr"),Ctr]),np.hstack([X18(Bc,"vl"),Cvl]),np.hstack([X18(Bc,"te"),Cte]))
Xtr_e=np.hstack([X18(Bc,"tr"),X18(Br,"tr"),conf_ent(Bc,"tr"),Ctr]); Xvl_e=np.hstack([X18(Bc,"vl"),X18(Br,"vl"),conf_ent(Bc,"vl"),Cvl]); Xte_e=np.hstack([X18(Bc,"te"),X18(Br,"te"),conf_ent(Bc,"te"),Cte])
P["stack_enriquecido"]=meta(Xtr_e,Xvl_e,Xte_e,C=0.03)
print("stacking listo:",{k:round(macroP(P[k][1],y_te,m_te),4) for k in ["stack_LR","stack_enriquecido"]})

In [ ]:
# CELDA 6 · MIXTURE-OF-EXPERTS (gating por paciente)
gsc=StandardScaler().fit(np.hstack([X18(Bc,"tr"),Ctr]))
def Gm(key,B): return gsc.transform(np.hstack([X18(B,key),{'tr':Ctr,'vl':Cvl,'te':Cte}[key]])).astype(np.float32)
class MoE(nn.Module):
    def __init__(s,d): super().__init__(); s.g=nn.Sequential(nn.Linear(d,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,3))
    def forward(s,g,cx,ec,la):
        w=F.softmax(s.g(g),1); return (w[:,0:1]*cx+w[:,1:2]*ec+w[:,2:3]*la).clamp(1e-6,1-1e-6)
def run_moe():
    mdl=MoE(Ctr.shape[1]+18); opt=torch.optim.AdamW(mdl.parameters(),lr=3e-3,weight_decay=1e-3)
    gt=torch.tensor(Gm("tr",Bc)); cx,ec,la=[torch.tensor(Bc["tr"][k]) for k in MODS]
    yt=torch.tensor(y_tr); mt=torch.tensor(m_tr); gv=torch.tensor(Gm("vl",Bc)); best=-1; bs=None; wait=0
    for ep in range(300):
        mdl.train(); perm=torch.randperm(len(gt))
        for i in range(0,len(gt),256):
            idx=perm[i:i+256]
            if len(idx)<2: continue
            opt.zero_grad(); p=mdl(gt[idx],cx[idx],ec[idx],la[idx]); bce=F.binary_cross_entropy(p,yt[idx],reduction="none")*mt[idx]
            (bce.sum()/mt[idx].sum().clamp(min=1e-8)).backward(); opt.step()
        mdl.eval()
        with torch.no_grad(): pv=mdl(gv,*[torch.tensor(Bc["vl"][k]) for k in MODS]).numpy()
        sc=macroP(pv,y_vl,m_vl)
        if sc>best+1e-4: best,bs,wait=sc,copy.deepcopy(mdl.state_dict()),0
        else:
            wait+=1
            if wait>=20: break
    if bs: mdl.load_state_dict(bs)
    mdl.eval()
    with torch.no_grad():
        return (mdl(torch.tensor(Gm("vl",Bc)),*[torch.tensor(Bc["vl"][k]) for k in MODS]).numpy(),
                mdl(torch.tensor(Gm("te",Bc)),*[torch.tensor(Bc["te"][k]) for k in MODS]).numpy())
P["MoE"]=run_moe(); print("MoE:",round(macroP(P["MoE"][1],y_te,m_te),4))

In [ ]:
# CELDA 7 · META-COMBINADORES: ensemble de fusores, selección por etiqueta, Caruana
cand={"CXR":(Bc["vl"]["CXR"],Bc["te"]["CXR"]), **{k:v for k,v in P.items()}}
# A) ensemble de los 3 mejores fusores (en val)
top=sorted(P,key=lambda k:macroP(P[k][0],y_vl,m_vl),reverse=True)[:3]
P["ensemble_fusores"]=(np.mean([P[k][0] for k in top],0),np.mean([P[k][1] for k in top],0))
# B) selección por etiqueta (mejor candidato según val)
sel_vl=np.zeros_like(Bc["vl"]["CXR"]); sel_te=np.zeros_like(Bc["te"]["CXR"]); pick={}
for j,l in enumerate(LABELS):
    s=m_vl[:,j]==1; yj=y_vl[s,j]; bestk,bestv=None,-1
    for k,(pv,pt) in cand.items():
        if yj.sum()<2 or (1-yj).sum()<2: continue
        a=roc_auc_score(yj,pv[s,j])
        if a>bestv: bestv,bestk=a,k
    pick[l]=bestk; sel_vl[:,j]=cand[bestk][0][:,j]; sel_te[:,j]=cand[bestk][1][:,j]
P["sel_por_etiqueta"]=(sel_vl,sel_te)
# C) Caruana hill-climbing por etiqueta (selección en val, con reemplazo)
def caruana(j, iters=40):
    s=m_vl[:,j]==1; yj=y_vl[s,j]
    if yj.sum()<2 or (1-yj).sum()<2: return cand["CXR"][0][:,j],cand["CXR"][1][:,j]
    keys=list(cand); best0=max(keys,key=lambda k:roc_auc_score(yj,cand[k][0][s,j])); chosen=[best0]
    cur_vl=cand[best0][0][:,j].copy(); cur_te=cand[best0][1][:,j].copy()
    for _ in range(iters):
        bk,bv=None,roc_auc_score(yj,cur_vl[s])
        for k in keys:
            mix=(cur_vl*len(chosen)+cand[k][0][:,j])/(len(chosen)+1)
            a=roc_auc_score(yj,mix[s])
            if a>bv+1e-6: bv,bk=a,k
        if bk is None: break
        chosen.append(bk); cur_vl=(cur_vl*(len(chosen)-1)+cand[bk][0][:,j])/len(chosen); cur_te=(cur_te*(len(chosen)-1)+cand[bk][1][:,j])/len(chosen)
    return cur_vl,cur_te
car_vl=np.zeros_like(Bc["vl"]["CXR"]); car_te=np.zeros_like(Bc["te"]["CXR"])
for j in range(N): car_vl[:,j],car_te[:,j]=caruana(j)
P["caruana"]=(car_vl,car_te)
print("Selección por etiqueta:",pick)

In [ ]:
# CELDA 8 · COMPARATIVA (elección por VAL) + guardado + figuras
order=["CXR","blend_opt(v3)","blend_logit","blend_raw","stack_LR","stack_enriquecido","MoE","ensemble_fusores","sel_por_etiqueta","caruana"]
res={}; print(f"{'Enfoque':20s} {'val':>8} {'TEST':>9}"); print("-"*40)
for k in order:
    pv,pt = (Bc["vl"]["CXR"],Bc["te"]["CXR"]) if k=="CXR" else P[k]
    res[k]={"val":macroP(pv,y_vl,m_vl),"test":macroP(pt,y_te,m_te),"per":aucs(pt,y_te,m_te)}
    print(f"{k:20s} {res[k]['val']:>8.4f} {res[k]['test']:>9.4f}")
bestk=max([k for k in order if k!="CXR"],key=lambda k:res[k]["val"])
print(f"\n>>> Elegido por VAL: {bestk} -> TEST macroP={res[bestk]['test']:.4f} (CXR={res['CXR']['test']:.4f}, v3={res['blend_opt(v3)']['test']:.4f})")
pd.DataFrame([{"enfoque":k,"macroP_val":res[k]["val"],"macroP_test":res[k]["test"],**{l:res[k]["per"][l] for l in LABELS}} for k in order]).to_csv(OUT/"comparativa_fusion_v4.csv",index=False)
json.dump({"best_by_val":bestk,"res":{k:{"val":res[k]["val"],"test":res[k]["test"]} for k in order},"pick":pick},open(OUT/"summary_v2.json","w",encoding="utf-8"),indent=2,ensure_ascii=False)
cols=["#95a5a6" if k=="CXR" else ("#c0392b" if k==bestk else "#2980b9") for k in order]; vals=[res[k]["test"] for k in order]
fig,ax=plt.subplots(figsize=(11,5)); ax.bar(order,vals,color=cols,alpha=0.9); ax.set_ylim(0.5,max(vals)+0.015)
ax.axhline(res["CXR"]["test"],color="gray",ls="--",label="CXR solo"); ax.axhline(res["blend_opt(v3)"]["test"],color="#27ae60",ls=":",label="v3 blend_opt")
ax.set_ylabel("macro AUC (patologías, test)"); ax.set_title("v4 — técnicas para exprimir el AUC (ganador en rojo)")
for i,v in enumerate(vals): ax.text(i,v+0.0015,f"{v:.4f}",ha="center",fontsize=8)
ax.legend(); plt.xticks(rotation=25,ha="right"); plt.tight_layout(); plt.savefig(FG/"comparativa_v4.png",dpi=150,bbox_inches="tight"); plt.show()
fig,ax=plt.subplots(figsize=(11,5)); x=np.arange(N); wb=0.26
for i,(k,c) in enumerate([("CXR","#95a5a6"),("blend_opt(v3)","#27ae60"),(bestk,"#c0392b")]):
    ax.bar(x+(i-1)*wb,[res[k]["per"][l] for l in LABELS],wb,label=k,color=c,alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels([l[:9] for l in LABELS],rotation=25,ha="right"); ax.axhline(0.5,color="gray",ls="--")
ax.set_ylabel("AUC (test)"); ax.set_title(f"AUC por etiqueta — CXR vs v3 vs v4 ({bestk})"); ax.legend()
plt.tight_layout(); plt.savefig(FG/"auc_etiqueta_v4.png",dpi=150,bbox_inches="tight"); plt.show()
print("Guardado salidas/04_stacking/v2/ (csv, json, figuras)")

# ══════════════════════════════════════════════════════════════════════════════
# ¿LA FUSIÓN SUPERA AL MEJOR MONO-MODELO?  (pregunta central del TFM)
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : compara el mejor fusor (elegido por VAL) contra la mejor modalidad EN SOLITARIO,
#              con IC bootstrap del AUC-PR macro, y declara si los intervalos SE SOLAPAN.
# POR QUÉ    : es la hipótesis que justifica todo el trabajo multimodal. Sin IC, una diferencia de
#              milésimas sobre 464 pacientes no demuestra nada.
# ORIGEN EDA : §11 · "comparar fusión vs mejor mono-modelo" · §5/§8 · "las modalidades por separado
#              no separan los diagnósticos; la fusión es necesaria" · §1 · "IC bootstrap siempre".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              si los IC se solapan, escribir explícitamente que la mejora NO es concluyente con este
#              tamaño de test. Es un resultado honesto y perfectamente defendible.
# ══════════════════════════════════════════════════════════════════════════════
MONO = {"CXR": (Bc["vl"]["CXR"], Bc["te"]["CXR"]),
        "ECG": (Bc["vl"]["ECG"], Bc["te"]["ECG"]),
        "LABS": (Bc["vl"]["LABS"], Bc["te"]["LABS"])}
mono_ap = {k: macroP(v[1], y_te, m_te) for k, v in MONO.items()}
best_mono = max(mono_ap, key=mono_ap.get)
ci_mono = ci_macroP(MONO[best_mono][1], y_te, m_te)
ap_fus = res[bestk]["test"]
ci_fus = ci_macroP((Bc["te"]["CXR"] if bestk == "CXR" else P[bestk][1]), y_te, m_te)
solapan = not (ci_fus[0] > ci_mono[1] or ci_mono[0] > ci_fus[1])

print("\n" + "=" * 78)
print("PREGUNTA CENTRAL: ¿aporta fusionar frente a usar la mejor modalidad sola?")
print("-" * 78)
for k, a in sorted(mono_ap.items(), key=lambda x: -x[1]):
    print(f"   mono {k:5s} macroAP(test) = {a:.4f}")
print(f"\n   MEJOR FUSOR      : {bestk:20s} macroAP={ap_fus:.4f}  IC95%=[{ci_fus[0]:.3f},{ci_fus[1]:.3f}]")
print(f"   MEJOR MONO-MODELO: {best_mono:20s} macroAP={mono_ap[best_mono]:.4f}  IC95%=[{ci_mono[0]:.3f},{ci_mono[1]:.3f}]")
print(f"   Diferencia       : {ap_fus - mono_ap[best_mono]:+.4f}")
print("=" * 78)
if solapan:
    print("VEREDICTO: los IC SE SOLAPAN -> la mejora de la fusion NO es concluyente.")
    print("           Con test=464 no puede afirmarse que fusionar supere a la mejor modalidad sola.")
else:
    print("VEREDICTO: los IC NO se solapan -> la fusion supera de forma consistente al mejor mono.")
print("RECORDATORIO: la AP no es comparable entre etiquetas (linea base = prevalencia de cada una).")

json.dump({"mejor_fusor": bestk, "macro_ap_fusion": ap_fus, "ic_fusion": list(ci_fus),
           "mejor_mono": best_mono, "macro_ap_mono": mono_ap[best_mono], "ic_mono": list(ci_mono),
           "macro_ap_por_mono": mono_ap, "diferencia": ap_fus - mono_ap[best_mono],
           "ic_se_solapan": bool(solapan),
           "veredicto": ("mejora NO concluyente (IC solapados)" if solapan else "la fusion supera al mejor mono"),
           "metrica": "AUC-PR macro sobre las 5 patologias (primaria)"},
          open(OUT / "veredicto_fusion_vs_mono.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
print("\nGuardado veredicto_fusion_vs_mono.json")




---
## ✅ Conclusión — v4

Exprimiendo el ranking sobre las mismas predicciones base, el AUC sube de **0.7825 (CXR)** y **0.7914 (v3 blend_opt)** a
**~0.794–0.795**. Lo que más aporta: combinar en **logit** y con **probabilidades raw** (la calibración aplanaba el
ranking), y sobre todo el **ensemble Caruana**, que selecciona y promedia los mejores fusores por etiqueta. La selección
final se hace **en validación** (honesta) y generaliza a test.

**Techo:** seguimos en fusión tardía sobre probabilidades (~+0.012 sobre el CXR). Para saltos mayores haría falta fusión
intermedia de *embeddings* (atención cruzada / contrastivo), que requiere GPU.
